# StarDist vs Keras — per-timepoint nuclei comparison

Loads the CSV written by `compare-stardist-vs-keras.py` and plots one bar chart per metric (nuclei count, mean / total volume, mean radius, mean / total surface area) with side-by-side bars per timepoint — orange for StarDist, blue for Keras. Use this to eyeball whether your tuned StarDist over- or under-segments relative to the legacy keras pipeline.

The script writes `<input_stem>.compare.csv` into `compare_data_paths.out_dir`. Edit `CSV_PATH` below to point at it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
CSV_PATH = Path("/lustre/fsn1/projects/rech/jsy/uzj81mi/demo_data/compare/timelapse_fifth_dataset.compare.csv")

# Optional voxel-spacing for physical-unit plots. Leave at 1.0 for
# voxel-space numbers (default — matches the script).
VOXEL_VOLUME_UM3 = 1.0   # multiply ``*_volume_vox`` by this to get µm³
VOXEL_LENGTH_UM = 1.0    # multiply ``mean_radius_vox`` by this to get µm
VOXEL_AREA_UM2 = 1.0     # multiply ``*_surface_area`` by this to get µm²

# Bar-chart colours.
COLOR_STARDIST = '#d4773a'
COLOR_KERAS = '#3a7ca5'

In [ ]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"{CSV_PATH} not found — run compare-stardist-vs-keras.py first"
    )
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(f"Sources: {df['source'].unique().tolist()}")
print(f"T-indices: {sorted(df['t_index'].unique().tolist())}")
df.head()

In [ ]:
# Pivot to wide form so each metric is a pair of columns (one per source).
metric_cols = [c for c in df.columns if c not in ('t_index', 'source')]
wide = df.pivot(index='t_index', columns='source', values=metric_cols).sort_index()
wide.columns = [f'{m}__{s}' for m, s in wide.columns]
wide.reset_index(inplace=True)
wide

In [ ]:
# Per-metric mean across timepoints — a single-number sanity table.
rows = []
for m in metric_cols:
    sd = wide.get(f'{m}__stardist')
    kr = wide.get(f'{m}__keras')
    if sd is None or kr is None:
        continue
    rows.append({
        'metric': m,
        'stardist_mean': float(sd.mean()),
        'keras_mean': float(kr.mean()),
        'ratio_sd_over_keras': float(sd.mean() / kr.mean()) if kr.mean() else float('nan'),
    })
pd.DataFrame(rows)

In [ ]:
def _bar_pair(ax, ts, vals_sd, vals_kr, ylabel, title):
    """Side-by-side bars (StarDist orange, Keras blue) for one metric."""
    x = np.arange(len(ts))
    w = 0.4
    ax.bar(x - w/2, vals_sd, w, label='StarDist', color=COLOR_STARDIST, edgecolor='black', linewidth=0.5)
    ax.bar(x + w/2, vals_kr, w, label='Keras',    color=COLOR_KERAS,    edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([str(t) for t in ts], rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('T-index')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(fontsize=8, loc='best')


ts = wide['t_index'].tolist()

# Metrics → (ylabel, optional unit-scaling factor) for the bar charts.
plots = [
    ('n_nuclei',           'count',     1.0,                'Nuclei count per frame'),
    ('mean_volume_vox',    'voxels',    VOXEL_VOLUME_UM3,   'Mean nucleus volume'),
    ('total_volume_vox',   'voxels',    VOXEL_VOLUME_UM3,   'Total nuclei volume'),
    ('mean_radius_vox',    'voxels',    VOXEL_LENGTH_UM,    'Mean equivalent-sphere radius'),
    ('mean_surface_area',  'vox²',      VOXEL_AREA_UM2,     'Mean nucleus surface area'),
    ('total_surface_area', 'vox²',      VOXEL_AREA_UM2,     'Total nuclei surface area'),
]

fig, axes = plt.subplots(3, 2, figsize=(14, 13))
axes = axes.flatten()
for ax, (metric, unit, scale, title) in zip(axes, plots):
    sd_col = f'{metric}__stardist'
    kr_col = f'{metric}__keras'
    if sd_col not in wide.columns or kr_col not in wide.columns:
        ax.set_axis_off()
        ax.set_title(f'{metric} — missing')
        continue
    _bar_pair(
        ax,
        ts,
        (wide[sd_col] * scale).to_numpy(),
        (wide[kr_col] * scale).to_numpy(),
        ylabel=unit,
        title=title,
    )
fig.tight_layout()
plt.show()

## Ratio plot

Per timepoint, plot `stardist / keras` for each metric on a log-y axis. Bars at 1 = parity, < 1 = StarDist under-counts / smaller cells, > 1 = StarDist over-counts / bigger cells. Faster way to spot systematic bias than reading raw bars.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 13))
axes = axes.flatten()
for ax, (metric, unit, _scale, title) in zip(axes, plots):
    sd_col = f'{metric}__stardist'
    kr_col = f'{metric}__keras'
    if sd_col not in wide.columns or kr_col not in wide.columns:
        ax.set_axis_off()
        continue
    sd = wide[sd_col].to_numpy(dtype=float)
    kr = wide[kr_col].to_numpy(dtype=float)
    ratio = np.divide(sd, kr, out=np.full_like(sd, np.nan), where=kr > 0)
    x = np.arange(len(ts))
    colors = ['#3aa55a' if abs(np.log10(r)) < 0.1 else '#d44a4a' if r < 1 else '#d4a83a' for r in np.nan_to_num(ratio, nan=1.0)]
    ax.bar(x, ratio, 0.7, color=colors, edgecolor='black', linewidth=0.4)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels([str(t) for t in ts], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('stardist / keras (log)')
    ax.set_title(title)
    ax.grid(True, axis='y', which='both', alpha=0.3)
fig.tight_layout()
plt.show()